# 02 — Latent Market Regime Identification & Volatility Dynamics

## Methodology
The natural gas storage-price elasticity is fundamentally regime-dependent. During normal liquidity conditions, storage inventory adjustments exert moderate influence on prices. In crisis periods, depleted inventories induce inelastic demand bidding and extreme volatility spikes.

We specify a **Two-Regime Markov Switching Autoregressive/Variance model** on TTF log-returns:
$$r_t = \mu_{S_t} + \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, \sigma^2_{S_t})$$
where $S_t \in \{0, 1\}$ represents the latent market state.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.features.regime_detection import RegimeDetector

df_features = pd.read_parquet('data/processed/feature_matrix.parquet')
print('Feature matrix columns:', df_features.columns.tolist()[:10])
df_features[['date', 'ttf_price', 'ttf_log_return', 'regime_label', 'prob_crisis_filtered', 'prob_crisis_smoothed']].head()

### 2.1 Causal Filtered vs Retrospective Smoothed Probabilities
To prevent look-ahead bias in predictive models, **filtered probabilities** $P(S_t = 1 | y_{1:t})$ are computed using only data up to day $t$. Full-sample **smoothed probabilities** $P(S_t = 1 | y_{1:T})$ are reserved for historical analysis.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
dates = pd.to_datetime(df_features['date'])

ax1.plot(dates, df_features['ttf_log_return'], color='forestgreen', alpha=0.7, lw=0.8, label='TTF Log Return')
ax1.set_ylabel('Log Return')
ax1.legend(loc='upper left')

ax2.plot(dates, df_features['prob_crisis_smoothed'], color='crimson', lw=1.8, label='Smoothed Prob P(S_t=Crisis | y_{1:T})')
ax2.plot(dates, df_features['prob_crisis_filtered'], color='steelblue', ls='--', lw=1.2, label='Filtered Prob P(S_t=Crisis | y_{1:t})')
ax2.axhline(0.5, color='gray', ls=':')
ax2.set_ylabel('Crisis Probability')
ax2.legend(loc='upper left')
plt.tight_layout()
plt.show()

### 2.2 Volatility Across Regimes
Comparing the realized volatility distribution between the identified Normal and Crisis states:

In [ ]:
regime_summary = df_features.groupby('regime_label')[['ttf_price', 'realized_vol_60d']].agg(['count', 'mean', 'std'])
display(regime_summary)